<a href="https://colab.research.google.com/github/SKaiForge/amt/blob/main/amt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi

Sun Nov 15 09:21:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
!cp -r /content/drive/MyDrive/huggingface-cache /content/huggingface-cache

In [4]:
# !cp -r /content/huggingface-cache /content/drive/MyDrive/huggingface-cache

In [5]:
import os

# persistent_cache = "/content/drive/MyDrive/huggingface-cache"

persistent_cache = "/content/huggingface-cache"


os.environ["HF_HOME"] = persistent_cache
os.environ["HUGGINGFACE_HUB_CACHE"] = persistent_cache
os.environ["TRANSFORMERS_CACHE"] = persistent_cache
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

In [6]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_properties(0).total_memory / 1024**3)

2.14.1+cu130
True
Tesla T4
14.56317138671875


In [7]:
from pathlib import Path
from google.colab import userdata
import os
import subprocess

token = userdata.get("GITHUB_TOKEN")

netrc = Path.home() / ".netrc"
netrc.write_text(
    f"machine github.com login x-access-token password {token}\n",
    encoding="utf-8",
)
netrc.chmod(0o600)

try:
    # Added capture_output=True and text=True to see the actual Git error
    result = subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/SKaiForge/amt.git",
            "/content/amt",
        ],
        capture_output=True,
        text=True,
        check=True,
    )
except subprocess.CalledProcessError as e:
    print("Git Clone Failed!")
    print("STDOUT:", e.stdout)
    print("STDERR:", e.stderr)
    raise e
finally:
    netrc.unlink(missing_ok=True)

Git Clone Successful!


In [8]:
import sys

repo_src = "/content/amt/src"
if repo_src not in sys.path:
    sys.path.insert(0, repo_src)

In [9]:
!pip install -U \
    "transformers>=4.51.0" \
    accelerate \
    safetensors \
    sentencepiece


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 46.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 66.4 MB/s  0:00:00
  Attempting uninstall: sentencepiece
    Found existing installation: sentencepiece 0.2.2
    Uninstalling sentencepiece-0.2.2:
      Successfully uninstalled sentencepiece-0.2.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.22.1
    Uninstalling transformers-5.22.1:
      Successfully uninstalled transformers-5.22.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [transformers]


In [10]:
pip install -U --no-deps bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 95.0 MB/s  0:00:00


In [11]:
import torch
import transformers
import bitsandbytes

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Transformers:", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

Torch: 2.14.1+cu130
CUDA available: True
Transformers: 5.23.0
GPU: Tesla T4


In [12]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

# Point to local cache on fast disk instead of slow Google Drive mount
cache_dir = "/content/huggingface-cache"

model_id = "Qwen/Qwen3-8B"
dtype = (
    torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
)

# Corrected the typo 'qunatization_config' to 'quantization_config'
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=dtype,
)

tokenizer = AutoTokenizer.from_pretrained(model_id, cache_dir=cache_dir)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto",
    cache_dir=cache_dir,
)
model.eval()

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 4096)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=12288, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=12288, bias=False)
          (down_proj): Linear4bit(in_features=12288, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((4096,), eps=1e-06

In [13]:
messages = [
    {
        "role": "system",
        "content": "Answer using the supplied user memory.",
    },
    {
        "role": "user",
        "content": (
            "Memory:\n"
            "The user prefers Python for ML experiments.\n\n"
            "Question: What language should the example use?"
        ),
    },
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=64,
        do_sample=False,
    )

generated = outputs[0, inputs.input_ids.shape[1]:]
answer = tokenizer.decode(generated, skip_special_tokens=True)

print(answer)

The example should use Python.


In [3]:
# %cd /content/amt

# !PYTHONPATH=/content/amt/src python3 -m datasets.amt_memorybench amt-100

In [2]:
# %cd /content/amt

# !PYTHONPATH=/content/amt/src python3 -m datasets.amt_memorybench smoke

In [1]:
# %cd /content/amt

# !PYTHONPATH=/content/amt/src python3 -m datasets.amt_memorybench atomic-smoke

In [ ]:
%cd /content/amt

!PYTHONPATH=src python3 -m datasets.amt_memorybench atomic-smoke-v2

In [17]:
# from pathlib import Path
# from datasets.amt_memorybench.answer import TextMemoryAnswerer
# from datasets.amt_memorybench.evaluate import (
#     LexicalMemorySelector,
#     read_jsonl,
# )
# from datasets.amt_memorybench.answer import evaluate_answers


# class QwenTextModel:
#     def __init__(self, model, tokenizer):
#         self.model = model
#         self.tokenizer = tokenizer

#     def generate(self, prompt: str) -> str:
#         messages = [{"role": "user", "content": prompt}]

#         text = self.tokenizer.apply_chat_template(
#             messages,
#             tokenize=False,
#             add_generation_prompt=True,
#             enable_thinking=False,
#         )
#         print(f"prompt: {prompt}")
#         inputs = self.tokenizer(text, return_tensors="pt").to(self.model.device)

#         with torch.inference_mode():
#             outputs = self.model.generate(
#                 **inputs,
#                 max_new_tokens=64,
#                 do_sample=False,
#             )

#         generated = outputs[0, inputs.input_ids.shape[1]:]
#         answer = self.tokenizer.decode(generated, skip_special_tokens=True)
#         print(f"answer: {answer}")
#         return answer


# qwen_model = QwenTextModel(model, tokenizer)
# answerer = TextMemoryAnswerer(qwen_model)

# # Wrapped the string path with Path() and pointed it to the cloned repository folder
# examples = read_jsonl(
#     Path("/content/amt/data/processed/amt_memorybench/amt-smoke.jsonl")
# )

# metrics = evaluate_answers(
#     examples,
#     LexicalMemorySelector(),
#     answerer,
#     k=1,
# )

# print(metrics)

In [18]:
from pathlib import Path
import torch

from amt_memory.models import QwenPrefixAMT
from datasets.amt_memorybench.evaluate import read_jsonl

amt = QwenPrefixAMT(model, tokenizer)

# Confirms Qwen is frozen and only the AMT projector is trainable.
trainable_parameters = sum(
    parameter.numel()
    for parameter in amt.parameters()
    if parameter.requires_grad
)

qwen_trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print("AMT trainable parameters:", trainable_parameters)
print("Qwen trainable parameters:", qwen_trainable_parameters)

AMT trainable parameters: 33562624
Qwen trainable parameters: 0


In [19]:
# examples = read_jsonl(
#     Path("/content/amt/data/processed/amt_memorybench/amt-smoke.jsonl")
# )

# single_recall = [
#     example
#     for example in examples
#     if example.task_type == "single_recall"
# ]

# def gold_postulates(example):
#     required_ids = set(example.required_memory_ids)
#     return [
#         memory.text
#         for memory in example.memories
#         if memory.memory_id in required_ids
#     ]

# def amt_prompt(question):
#     return (
#         "Answer with only the shortest answer span. "
#         "Do not explain or add advice.\n\n"
#         f"Question: {question}\nAnswer:"
#     )

In [20]:
# train_examples = [
#     example for example in single_recall
#     if example.example_id.endswith("_00")
# ]

# test_examples = [
#     example for example in single_recall
#     if example.example_id.endswith("_01")
# ]

# len(train_examples), len(test_examples)

In [21]:
# import torch
# from pathlib import Path
# from datasets.amt_memorybench.evaluate import read_jsonl

# # 1. Reset any monkey-patched forward methods on the base model
# if 'model' in globals():
#     try:
#         del model.forward
#     except AttributeError:
#         pass

# # 2. Re-initialize the AMT wrapper to completely wipe the existing NaN-corrupted parameters
# if 'amt' in globals():
#     del amt

# from amt_memory.models import QwenPrefixAMT
# amt = QwenPrefixAMT(model, tokenizer)

# # 3. Load dataset and prepare training/testing splits
# examples = read_jsonl(
#     Path("/content/amt/data/processed/amt_memorybench/amt-smoke.jsonl")
# )

# single_recall = [
#     example
#     for example in examples
#     if example.task_type == "single_recall"
# ]

# def gold_postulates(example):
#     required_ids = set(example.required_memory_ids)
#     return [
#         memory.text
#         for memory in example.memories
#         if memory.memory_id in required_ids
#     ]

# def amt_prompt(question):
#     return (
#         "Answer with only the shortest answer span. "
#         "Do not explain or add advice.\n\n"
#         f"Question: {question}\nAnswer:"
#     )

# train_examples = [
#     example for example in single_recall
#     if example.example_id.endswith("_00")
# ]

# test_examples = [
#     example for example in single_recall
#     if example.example_id.endswith("_01")
# ]

# # Ensure the base model vocabulary embeddings remain completely frozen
# if hasattr(amt, 'embedding'):
#     for param in amt.embedding.parameters():
#         param.requires_grad = False

# # 4. Extract ONLY the actual trainable projector module
# trainable_modules = []
# for attr_name in dir(amt):
#     attr = getattr(amt, attr_name)
#     if isinstance(attr, torch.nn.Module) and attr_name not in ["model", "embedding"]:
#         trainable_modules.append(attr)

# # Force the projector layers to be stored in highly stable FP32
# for module in trainable_modules:
#     module.to(dtype=torch.float32)

# # Intercept projector forward pass to safely run calculation in FP32
# original_projector_forward = amt.projector.forward

# def safe_projector_forward(x):
#     # 1. Upcast input activations to float32
#     x_f32 = x.to(torch.float32)
#     # 2. Compute projections in stable float32
#     out_f32 = original_projector_forward(x_f32)
#     # 3. Downcast back to the base model's precision (bfloat16 or float16)
#     return out_f32.to(dtype)

# amt.projector.forward = safe_projector_forward

# trainable_params = []
# for module in trainable_modules:
#     for param in module.parameters():
#         if param.requires_grad:
#             # Clean re-initialization of weights to wipe out the NaN pollution
#             torch.nn.init.normal_(param, mean=0.0, std=0.02)
#             trainable_params.append(param)

# # 5. Setup optimizer with standard learning rate for training the FP32 projector
# optimizer = torch.optim.AdamW(
#     trainable_params,
#     lr=1e-4,
#     weight_decay=0.01,
#     eps=1e-8
# )

# # 6. Training loop
# for module in trainable_modules:
#     module.train()

# for epoch in range(30):
#     total_loss = 0.0
#     successful_steps = 0

#     for example in train_examples:
#         optimizer.zero_grad(set_to_none=True)

#         loss = amt.loss(
#             postulates=gold_postulates(example),
#             prompt=amt_prompt(example.query),
#             answer=example.answer,
#         )

#         if torch.isnan(loss) or torch.isinf(loss):
#             continue

#         loss.backward()
#         # Clip gradients to keep updates extremely stable
#         torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=0.5)
#         optimizer.step()

#         total_loss += loss.item()
#         successful_steps += 1

#     if epoch % 5 == 0:
#         avg_loss = total_loss / successful_steps if successful_steps > 0 else float('nan')
#         print(f"epoch={epoch:02d} loss={avg_loss:.4f}")

In [22]:
# import re

# def normalize(text):
#     # Lowercase, replace non-alphanumeric characters with spaces, and split into words
#     cleaned = re.sub(r'[^a-zA-Z0-9\s]', ' ', text.lower())
#     return cleaned.split()

# correct = 0

# amt.eval()

# for example in test_examples:
#     prediction = amt.generate(
#         postulates=gold_postulates(example),
#         query=amt_prompt(example.query),
#         max_new_tokens=16,
#     )

#     # Get normalized text strings
#     expected_words = normalize(example.answer)
#     actual_words = normalize(prediction)

#     expected_str = " ".join(expected_words)
#     actual_str = " ".join(actual_words)

#     # Allow partial match (e.g., if gold is 'PyTorch for ML' and prediction contains 'pytorch')
#     # checking if key terms in expected are found in the actual prediction
#     is_correct = any(word in actual_words for word in expected_words if word not in ['for', 'ml', 'experiments'])
#     correct += is_correct

#     print("Question:", example.query)
#     print("Gold:", example.answer)
#     print("Prediction:", prediction.strip())
#     print("Correct:", bool(is_correct))
#     print()

# print("H1/H3 held-out-paraphrase accuracy:", correct / len(test_examples))

In [23]:
# import torch
# import torch.nn as nn

# print("=== Starting Diagnostics & Controls ===")

# # ---------------------------------------------------------
# # Control 1: Text-Memory Baseline Control
# # ---------------------------------------------------------
# print("\n--- Running Control: Text-Memory Baseline ---")
# prompt_text = (
#     "Answer with only the shortest answer span.\n\n"
#     "Memory: The user prefers Julia for ML experiments.\n"
#     "Question: Which framework is preferred?\n"
#     "Answer:"
# )

# inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
# with torch.inference_mode():
#     outputs = model.generate(**inputs, max_new_tokens=16, do_sample=False)
# text_generation = tokenizer.decode(outputs[0, inputs.input_ids.shape[1]:], skip_special_tokens=True)
# print(f"Text Control Generation: '{text_generation.strip()}'")

# # ---------------------------------------------------------
# # Experiment: Multi-Token Free-Vector Overfit (100 steps)
# # ---------------------------------------------------------
# print("\n--- Running Experiment: multiple-Token Free-Vector Overfit ---")
# token_count = 1
# hidden_size = model.config.hidden_size
# dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

# # Using 5 prefix tokens to guarantee capacity
# free_prefix = nn.Parameter(torch.randn(1, token_count, hidden_size, device=model.device, dtype=dtype) * 0.02)

# def amt_prompt(question):
#     return (
#         "Answer with only the shortest answer span. "
#         "Do not explain or add advice.\n\n"
#         f"Question: {question}\nAnswer:"
#     )

# question_prompt = amt_prompt("Which framework is preferred?")
# target_answer = " Julia"

# prompt_ids = tokenizer(question_prompt, return_tensors="pt", add_special_tokens=False).input_ids.to(model.device)
# answer_ids = tokenizer(target_answer, return_tensors="pt", add_special_tokens=False).input_ids.to(model.device)
# eos = torch.tensor(
#     [[tokenizer.eos_token_id]],
#     dtype=torch.long,
#     device=model.device,
# )
# answer_ids = torch.cat([answer_ids, eos], dim=1)

# optimizer = torch.optim.AdamW([free_prefix], lr=1e-2)  # slightly higher LR for faster convergence

# for step in range(101):
#     optimizer.zero_grad(set_to_none=True)

#     with torch.no_grad():
#         prompt_embeds = model.get_input_embeddings()(prompt_ids)
#         answer_embeds = model.get_input_embeddings()(answer_ids)

#     prefix_for_model = free_prefix.to(prompt_embeds.dtype)

#     full_embeds = torch.cat(
#         [prefix_for_model, prompt_embeds, answer_embeds],
#         dim=1,
#     )

#     # Ignore prefix and prompt; supervise every answer token.
#     full_labels = torch.full(
#         full_embeds.shape[:2],
#         -100,
#         dtype=torch.long,
#         device=model.device,
#     )
#     answer_start = token_count + prompt_ids.shape[1]

#     full_labels[:, answer_start:] = answer_ids

#     outputs = model(
#         inputs_embeds=full_embeds,
#         attention_mask=torch.ones(
#             full_embeds.shape[:2],
#             dtype=torch.long,
#             device=model.device,
#         ),
#         labels=full_labels,
#     )

#     loss = outputs.loss
#     loss.backward()
#     torch.nn.utils.clip_grad_norm_([free_prefix], 1.0)
#     optimizer.step()

#     if step % 10 == 0:
#         print(step, loss.item(), free_prefix.grad.norm().item())

In [24]:
# prompt_embeds = model.get_input_embeddings()(prompt_ids)
# inputs_embeds = torch.cat([free_prefix, prompt_embeds], dim=1)

# attention_mask = torch.ones(
#     inputs_embeds.shape[:2],
#     dtype=torch.long,
#     device=model.device,
# )

# with torch.inference_mode():
#     generated = model.generate(
#         inputs_embeds=inputs_embeds,
#         attention_mask=attention_mask,
#         max_new_tokens=8,
#         do_sample=False,
#     )

# print(tokenizer.decode(generated[0], skip_special_tokens=True))

In [25]:
# test_prompt = amt_prompt("What ML framework does the user prefer?")
# test_ids = tokenizer(
#     test_prompt,
#     return_tensors="pt",
# ).input_ids.to(model.device)

# test_embeds = model.get_input_embeddings()(test_ids)
# test_inputs = torch.cat([free_prefix, test_embeds], dim=1)

# with torch.inference_mode():
#     generated = model.generate(
#         inputs_embeds=test_inputs,
#         attention_mask=torch.ones(
#             test_inputs.shape[:2],
#             dtype=torch.long,
#             device=model.device,
#         ),
#         max_new_tokens=8,
#         do_sample=False,
#     )

# print(tokenizer.decode(generated[0], skip_special_tokens=True))

In [26]:
# import torch

# embedding_layer = model.get_input_embeddings()
# model.eval()

# def amt_prompt(question: str) -> str:
#     return (
#         "Answer with only the shortest answer span. "
#         "Do not explain or add advice.\n\n"
#         f"Question: {question}\nAnswer:"
#     )


# def generate_from_prefix(prefix: torch.Tensor, question: str) -> str:
#     prompt = amt_prompt(question)

#     prompt_ids = tokenizer(
#         prompt,
#         return_tensors="pt",
#         add_special_tokens=True,
#     ).input_ids.to(model.device)

#     with torch.no_grad():
#         prompt_embeds = embedding_layer(prompt_ids)

#     prefix = prefix.to(
#         device=prompt_embeds.device,
#         dtype=prompt_embeds.dtype,
#     )

#     inputs_embeds = torch.cat(
#         [prefix, prompt_embeds],
#         dim=1,
#     )

#     attention_mask = torch.ones(
#         inputs_embeds.shape[:2],
#         dtype=torch.long,
#         device=model.device,
#     )

#     with torch.inference_mode():
#         generated = model.generate(
#             inputs_embeds=inputs_embeds,
#             attention_mask=attention_mask,
#             max_new_tokens=8,
#             do_sample=False,
#             pad_token_id=tokenizer.eos_token_id,
#         )

#     return tokenizer.decode(
#         generated[0],
#         skip_special_tokens=True,
#     ).strip()

In [27]:
# original_question = "Which framework is preferred?"
# paraphrase_question = "What ML framework does the user prefer?"

# print("=== Correct learned prefix ===")
# print("Original:", generate_from_prefix(free_prefix.detach(), original_question))
# print("Paraphrase:", generate_from_prefix(free_prefix.detach(), paraphrase_question))

In [28]:
# zero_prefix = torch.zeros_like(free_prefix)

# random_generator = torch.Generator(device=model.device)
# random_generator.manual_seed(17)

# random_prefix = torch.randn(
#     free_prefix.shape,
#     generator=random_generator,
#     device=free_prefix.device,
#     dtype=free_prefix.dtype,
# ) * 0.02

# print("\n=== Zero prefix ===")
# print(generate_from_prefix(zero_prefix, original_question))

# print("\n=== Random prefix ===")
# print(generate_from_prefix(random_prefix, original_question))

In [29]:
import gc
from collections import defaultdict, Counter

import torch
import torch.nn as nn

# ---------------------------------------------------------
# Load atomic benchmark examples
# ---------------------------------------------------------

from pathlib import Path
from datasets.amt_memorybench.evaluate import read_jsonl

benchmark_path = Path(
    "/content/amt/data/processed/amt_memorybench/amt-atomic-smoke.jsonl"
)

examples = read_jsonl(benchmark_path)

groups = defaultdict(list)

for example in examples:
    # Fallback to stripping the trailing _00/_01 suffix from example_id if paraphrase_group_id is None
    group_id = example.paraphrase_group_id
    if group_id is None:
        group_id = "_".join(example.example_id.split("_")[:-1])
    groups[group_id].append(example)

groups = {
    group_id: sorted(group, key=lambda example: example.example_id)
    for group_id, group in groups.items()
}

assert len(groups) == 60, f"Expected 60 groups, got {len(groups)}"
assert all(len(group) == 2 for group in groups.values()), "Each group must contain exactly 2 examples"

train_examples = [group[0] for group in groups.values()]
test_examples = [group[1] for group in groups.values()]

print("Groups:", len(groups))
print("Train examples:", len(train_examples))
print("Test examples:", len(test_examples))

Groups: 60
Train examples: 60
Test examples: 60


In [30]:
embedding_layer = model.get_input_embeddings()
hidden_size = model.config.hidden_size

model.eval()

for parameter in model.parameters():
    parameter.requires_grad_(False)


def prompt_for(question: str) -> str:
    return (
        "Answer with only the shortest answer span. "
        "Do not explain or add advice.\n\n"
        f"Question: {question}\n"
        "Answer:"
    )


def normalize(text: str) -> str:
    return " ".join(text.lower().strip().split())


def build_answer_ids(answer: str):
    # The leading space represents continuation after "Answer:".
    answer_ids = tokenizer(
        " " + answer,
        return_tensors="pt",
        add_special_tokens=False,
    ).input_ids.to(model.device)

    # Teach the model to stop after the answer.
    if tokenizer.eos_token_id is not None:
        eos = torch.tensor(
            [[tokenizer.eos_token_id]],
            dtype=torch.long,
            device=model.device,
        )
        answer_ids = torch.cat([answer_ids, eos], dim=1)

    return answer_ids


def generate_from_prefix(prefix, question, max_new_tokens=16):
    prompt = prompt_for(question)

    prompt_ids = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=True,
    ).input_ids.to(model.device)

    with torch.no_grad():
        prompt_embeds = embedding_layer(prompt_ids)

    prefix_for_model = prefix.to(
        device=prompt_embeds.device,
        dtype=prompt_embeds.dtype,
    )

    inputs_embeds = torch.cat(
        [prefix_for_model, prompt_embeds],
        dim=1,
    )

    attention_mask = torch.ones(
        inputs_embeds.shape[:2],
        dtype=torch.long,
        device=model.device,
    )

    with torch.inference_mode():
        generated = model.generate(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    return tokenizer.decode(
        generated[0],
        skip_special_tokens=True,
    ).strip()

def train_one_free_token(
    example,
    token_count=1,
    steps=100,
    learning_rate=1e-2,
):
    torch.manual_seed(17)

    # Keep optimizer parameters in FP32.
    free_prefix = nn.Parameter(
        torch.randn(
            1,
            token_count,
            hidden_size,
            device=model.device,
            dtype=torch.float32,
        ) * 0.02
    )

    optimizer = torch.optim.AdamW(
        [free_prefix],
        lr=learning_rate,
        eps=1e-6,
        weight_decay=0.0,
    )

    prompt = prompt_for(example.query)

    prompt_ids = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=True,
    ).input_ids.to(model.device)

    answer_ids = build_answer_ids(example.answer)

    with torch.no_grad():
        prompt_embeds = embedding_layer(prompt_ids)
        answer_embeds = embedding_layer(answer_ids)

    # Qwen may use BF16/FP16, while the free prefix remains FP32.
    prompt_embeds = prompt_embeds.detach()
    answer_embeds = answer_embeds.detach()

    losses = []

    for step in range(steps):
        optimizer.zero_grad(set_to_none=True)

        prefix_for_model = free_prefix.to(
            dtype=prompt_embeds.dtype,
            device=prompt_embeds.device,
        )

        full_embeds = torch.cat(
            [
                prefix_for_model,
                prompt_embeds,
                answer_embeds,
            ],
            dim=1,
        )

        prefix_length = free_prefix.shape[1]
        prompt_length = prompt_embeds.shape[1]

        full_labels = torch.full(
            full_embeds.shape[:2],
            -100,
            dtype=torch.long,
            device=model.device,
        )

        answer_start = prefix_length + prompt_length
        full_labels[:, answer_start:] = answer_ids

        attention_mask = torch.ones(
            full_embeds.shape[:2],
            dtype=torch.long,
            device=model.device,
        )

        outputs = model(
            inputs_embeds=full_embeds,
            attention_mask=attention_mask,
            labels=full_labels,
        )

        loss = outputs.loss

        if not torch.isfinite(loss):
            raise FloatingPointError(
                f"Non-finite loss at step {step}: {loss.item()}"
            )

        loss.backward()

        if free_prefix.grad is None:
            raise RuntimeError("Free prefix received no gradient")

        if not torch.isfinite(free_prefix.grad).all():
            raise FloatingPointError(
                f"Non-finite free-prefix gradient at step {step}"
            )

        torch.nn.utils.clip_grad_norm_(
            [free_prefix],
            max_norm=1.0,
        )

        optimizer.step()

        losses.append(loss.item())

    return free_prefix.detach(), losses

In [31]:
import time

results = []

for index, (train_example, test_example) in enumerate(
    zip(train_examples, test_examples),
    start=1,
):
    memory_type = train_example.memories[
        train_example.required_memory_ids[0]
        in [memory.memory_id for memory in train_example.memories]
    ].type if False else None

    started = time.perf_counter()
    print(f"[{index:02d}] Starting training", flush=True)

    train_prefix, losses = train_one_free_token(
        train_example,
        token_count=1,
        steps=100,
        learning_rate=1e-2,
    )

    print(
        f"[{index:02d}] Training finished "
        f"({time.perf_counter() - started:.1f}s); generating train",
        flush=True,
    )

    train_prediction = generate_from_prefix(
        train_prefix,
        train_example.query,
    )

    print(f"[{index:02d}] Generating test", flush=True)

    test_prediction = generate_from_prefix(
        train_prefix,
        test_example.query,
    )

    print(f"[{index:02d}] Both generations finished", flush=True)

    train_correct = normalize(train_example.answer) in normalize(
        train_prediction
    )

    test_correct = normalize(test_example.answer) in normalize(
        test_prediction
    )

    required_memory = next(
        memory
        for memory in train_example.memories
        if memory.memory_id in train_example.required_memory_ids
    )

    result = {
        "index": index,
        "memory_type": required_memory.type,
        "postulate": required_memory.text,
        "train_question": train_example.query,
        "test_question": test_example.query,
        "gold_answer": test_example.answer,
        "train_prediction": train_prediction,
        "test_prediction": test_prediction,
        "train_correct": train_correct,
        "test_correct": test_correct,
        "initial_loss": losses[0],
        "final_loss": losses[-1],
    }

    results.append(result)

    print(
        f"[{index:02d}/60] "
        f"type={required_memory.type:<12} "
        f"loss={losses[-1]:.4f} "
        f"train={train_correct} "
        f"test={test_correct}"
    )

    del train_prefix
    gc.collect()
    torch.cuda.empty_cache()

[01] Starting training
[01] Training finished (89.1s); generating train
[01] Generating test
[01] Both generations finished
[01/60] type=fact         loss=0.0003 train=True test=True
[02] Starting training
[02] Training finished (102.0s); generating train
[02] Generating test
[02] Both generations finished
[02/60] type=preference   loss=0.0003 train=True test=True
[03] Starting training
[03] Training finished (102.3s); generating train
[03] Generating test
[03] Both generations finished
[03/60] type=goal         loss=0.0003 train=True test=True
[04] Starting training
[04] Training finished (124.5s); generating train
[04] Generating test
[04] Both generations finished
[04/60] type=constraint   loss=0.0004 train=True test=True
[05] Starting training
[05] Training finished (102.1s); generating train
[05] Generating test
[05] Both generations finished
[05/60] type=event        loss=0.0006 train=True test=True
[06] Starting training
[06] Training finished (101.9s); generating train
[06] Gen

In [32]:
import json

with open("results.json", "w") as f:
    json.dump(results, f, indent=2)

In [33]:
!cp -r /content/amt/results.json /content/drive/MyDrive/amt-experiments/

In [34]:
overall_test_accuracy = sum(
    result["test_correct"]
    for result in results
) / len(results)

print("\nOverall paraphrase accuracy:", overall_test_accuracy)

type_results = defaultdict(list)

for result in results:
    type_results[result["memory_type"]].append(result)

print("\nAccuracy by memory type:")

for memory_type, type_items in sorted(type_results.items()):
    accuracy = sum(
        item["test_correct"]
        for item in type_items
    ) / len(type_items)

    print(
        f"{memory_type:<14} "
        f"{accuracy:.3f} "
        f"({sum(item['test_correct'] for item in type_items)}/{len(type_items)})"
    )


Overall paraphrase accuracy: 0.8666666666666667

Accuracy by memory type:
constraint     0.800 (8/10)
event          0.600 (6/10)
fact           1.000 (10/10)
goal           1.000 (10/10)
preference     1.000 (10/10)
relationship   0.800 (8/10)


In [35]:
for correct, heading in [(True, "SUCCESSES"), (False, "FAILURES")]:
    examples = [
        result
        for result in results
        if result["test_correct"] == correct
    ]

    print(f"\n{'=' * 70}")
    print(f"{heading}: {len(examples)}/{len(results)}")
    print("=" * 70)

    for result in examples:
        print(f"\n[{result['index']:02d}] {result['memory_type']}")
        print(f"Postulate:        {result['postulate']}")
        print(f"Train question:   {result['train_question']}")
        print(f"Train prediction: {result['train_prediction']!r}")
        print(f"Test question:    {result['test_question']}")
        print(f"Expected answer:  {result['gold_answer']!r}")
        print(f"Test prediction:  {result['test_prediction']!r}")
        print(f"Final loss:       {result['final_loss']:.4f}")
        print("-" * 70)


SUCCESSES: 52/60

[01] fact
Postulate:        The user knows Java.
Train question:   What programming language does the user know?
Train prediction: 'Java'
Test question:    Which language has the user learned?
Expected answer:  'Java'
Test prediction:  'Java'
Final loss:       0.0003
----------------------------------------------------------------------

[02] preference
Postulate:        The user prefers TensorFlow for ML experiments.
Train question:   What does the user prefer for ML experiments?
Train prediction: 'TensorFlow'
Test question:    For ML experiments, which option does the user prefer?
Expected answer:  'TensorFlow'
Test prediction:  'TensorFlow'
Final loss:       0.0003
----------------------------------------------------------------------

[03] goal
Postulate:        The user is learning distributed systems.
Train question:   What is the user currently learning?
Train prediction: 'distributed systems'
Test question:    Which subject is the user studying?
Expected answ